# EPCLM-007 sequential retrieval gates
Select Tesla T4 and Internet. The direct CPU gate must pass before any GPU training. Raw evidence stays in Kaggle Output.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import signal
import subprocess
import sys
import time

started=time.monotonic()
stop=started+10500
remote='https://github.com/ArcheLabs/mini-cells.git'
branch='research/epclm007-cell-retrieval-learning'
repo=Path('/kaggle/working/minicells-epclm007')
os.environ['OMP_NUM_THREADS']='1'
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'

def run(argv,limit):
    remaining=min(limit,stop-time.monotonic())
    if remaining<=0:
        raise TimeoutError('EPCLM-006 session budget exhausted')
    process=subprocess.Popen(argv,start_new_session=True)
    try:
        code=process.wait(timeout=remaining)
    except subprocess.TimeoutExpired:
        os.killpg(process.pid,signal.SIGTERM)
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid,signal.SIGKILL)
            process.wait()
        raise TimeoutError('EPCLM-006 stopped at session budget. Preserve complete Output.')
    if code:
        raise RuntimeError('EPCLM-006 stopped. Inspect the retained error and checkpoints.')

if not repo.exists():
    run(['git','clone','--depth=1','--branch',branch,remote,str(repo)],300)
commit=subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()
base=repo/'research/projects/cellular-language-models/experiments/epclm-007-cell-retrieval-learning'
requirements=base.parent/'epclm-004-unified-memory-10m/m4/requirements.txt'
out=Path('/kaggle/working/epclm007-'+commit[:12])
run([sys.executable,'-m','pip','install','-q','-r',str(requirements)],300)
if not out.exists():
    for candidate in Path('/kaggle/input').rglob(out.name+'/decision.json'):
        shutil.copytree(candidate.parent,out)
        break
os.chdir(repo)
run([sys.executable,str(base/'screen007.py'),str(out),'--started',str(started)],10500)
